# Q55: Handling Imbalanced Data
**Topic:** Classical-ml | **Difficulty:** Medium | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
How do you handle imbalanced datasets?

## Detailed Answer

### The Problem
When one class dominates (e.g., 99% negative, 1% positive), models learn to always predict the majority class and achieve high accuracy while being useless.

### Strategies

#### 1. Resampling
- **Oversampling minority**: SMOTE (Synthetic Minority Oversampling)
  - Creates synthetic samples by interpolating between minority neighbors
  - Variants: ADASYN (adaptive), Borderline-SMOTE
- **Undersampling majority**: Random or Tomek links (remove borderline majority samples)
- **Combination**: SMOTE + Tomek links or SMOTE + ENN (Edited Nearest Neighbors)

#### 2. Class Weights
- Increase loss weight for minority class: `class_weight='balanced'`
- Weight = $\frac{N}{K \cdot n_k}$ where $n_k$ = samples in class $k$
- Built into sklearn, PyTorch (`CrossEntropyLoss(weight=...)`)  

#### 3. Focal Loss
$$FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$
- Down-weights easy (well-classified) examples, focuses on hard ones
- Used in RetinaNet (object detection) with great success
- $\gamma=2$ is typical

#### 4. Evaluation Metrics
Don't use accuracy! Use:
- **Precision, Recall, F1** (per-class and weighted)
- **PR-AUC** (Precision-Recall curve) > ROC-AUC for imbalanced
- **Matthews Correlation Coefficient** (balanced metric)

#### 5. Algorithmic Approaches
- **Ensemble**: Balanced Random Forest, EasyEnsemble
- **Anomaly detection**: Treat minority as anomalies (One-Class SVM, IF)
- **Threshold tuning**: Move decision threshold from 0.5

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Imbalanced dataset
X, y = make_classification(n_samples=1000, weights=[0.95], n_informative=3,
                           n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Without handling
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print('No balancing:')
print(classification_report(y_test, rf.predict(X_test), digits=3))

# With class weights
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42).fit(X_train, y_train)
print('With class_weight=balanced:')
print(classification_report(y_test, rf_balanced.predict(X_test), digits=3))

## Key Takeaways
- **Class weights** is the simplest fix — always try `class_weight='balanced'` first
- Use **SMOTE** for moderate imbalance; for extreme imbalance, consider anomaly detection
- **Never use accuracy** for imbalanced data — use F1, PR-AUC, or MCC
- **Focal Loss** is the go-to for object detection (always imbalanced)